# 05 — Baseline Inference (Variants A, B, C) — Colab Version

Run the open-source baselines using **Qwen3-4B-Instruct** on Google Colab.

Data is loaded from [HuggingFace](https://huggingface.co/datasets/siavashsaki/wdc-pave-ave).
The `ave/` package is cloned from the repo for reuse.

| Variant | Description | Model | Decoding |
|---------|-------------|-------|----------|
| A | Zero-shot, greedy | Qwen3-4B-Instruct-2507 | Unconstrained |
| B | Zero-shot + constrained JSON | Qwen3-4B-Instruct-2507 | JSON Schema via `outlines` |
| C | Few-shot (3 examples), greedy | Qwen3-4B-Instruct-2507 | Unconstrained |

**Runtime:** Set to GPU (T4 or better) via Runtime > Change runtime type.

## 0. Setup

In [ ]:
!pip install -q torch transformers accelerate datasets huggingface_hub outlines jinja2 tqdm pandas sentencepiece protobuf

In [ ]:
# Clone repo to get the ave/ package
import os

REPO_URL = "https://github.com/siavash-saki/ave.git"  # UPDATE if different
REPO_DIR = "/content/open_source_on_prem_models"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

# Add repo root to Python path so `import ave` works
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import ave  # verify import works
print("ave package loaded from:", os.path.dirname(ave.__file__))

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import torch
import pandas as pd
from huggingface_hub import hf_hub_download

HF_REPO = "siavashsaki/wdc-pave-ave"
DATA_DIR = Path("data/cleaned")
PRED_DIR = Path("predictions")
DATA_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Model:  {MODEL_ID}")
if device != "cuda":
    print("WARNING: No GPU detected. Inference will be very slow.")

## 1. Download data from HuggingFace

In [ ]:
import shutil

# Download dataset files from HF and place them where load_splits() expects
for filename in ["train.jsonl", "val.jsonl", "test.jsonl", "category_schemas.json"]:
    cached = hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type="dataset")
    dest = DATA_DIR / filename
    if not dest.exists():
        shutil.copy(cached, dest)
    print(f"  {filename:30s} -> {dest}")

print("\nData ready.")

## 2. Load data and schemas

In [ ]:
from ave.data.loader import load_splits, index_by_category
from ave.configs.schemas import CATEGORY_SCHEMAS, all_json_schemas

train, val, test = load_splits(DATA_DIR)
train_by_cat = index_by_category(train)
JSON_SCHEMAS = all_json_schemas()

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(f"Categories: {list(CATEGORY_SCHEMAS.keys())}")

## 3. Load model and tokenizer

In [ ]:
from ave.inference._local import load_model

model, tokenizer = load_model(MODEL_ID)

## 4. Variant A — Zero-shot smoke test

In [ ]:
from ave.inference._local import generate_response
from ave.prompts.templates import build_messages_zero_shot

# One val record per category
smoke_records = []
for cat in CATEGORY_SCHEMAS:
    rec = next(r for r in val if r["category"] == cat)
    smoke_records.append(rec)

print(f"Smoke test: {len(smoke_records)} records (one per category)\n")

smoke_results = []
for rec in smoke_records:
    messages = build_messages_zero_shot(rec, CATEGORY_SCHEMAS)
    result = generate_response(model, tokenizer, messages)
    smoke_results.append((rec, result))

    print(f"=== {rec['category']} (ID={rec['id']}) ===")
    print(f"  Latency: {result['latency_s']}s | In: {result['input_tokens']} tok | Out: {result['output_tokens']} tok")
    if result["parse_error"]:
        print(f"  PARSE ERROR: {result['parse_error']}")
        print(f"  Raw output: {result['raw_output'][:300]}")
    else:
        print(f"  Parsed JSON keys: {list(result['parsed_json'].keys())}")
    print()

In [ ]:
# Compare predicted vs gold for smoke test
for rec, result in smoke_results:
    print(f"=== {rec['category']} (ID={rec['id']}) ===")
    gold = rec["gold_json"]
    pred = result["parsed_json"]

    if pred is None:
        print("  No parsed JSON — skipping comparison")
        print(f"  Raw: {result['raw_output'][:200]}")
        print()
        continue

    schema = CATEGORY_SCHEMAS[rec["category"]]
    print(f"  {'Attribute':<30s} {'Gold':<40s} {'Predicted':<40s} {'Match'}")
    print(f"  {'-'*30} {'-'*40} {'-'*40} {'-'*5}")
    for attr in schema:
        g = gold.get(attr)
        p = pred.get(attr)
        g_str = json.dumps(g) if g is not None else "null"
        p_str = json.dumps(p) if p is not None else "null"
        match = "OK" if g == p else "---"
        if match == "---" and isinstance(g, str) and isinstance(p, str) and g.lower() == p.lower():
            match = "~ci"
        print(f"  {attr:<30s} {g_str:<40s} {p_str:<40s} {match}")
    print()

## 5. Variant A — Full val set

In [ ]:
from ave.inference._local import run_inference
from ave.utils import save_jsonl

preds_a = run_inference(
    model, tokenizer,
    records=val,
    message_builder=lambda rec: build_messages_zero_shot(rec, CATEGORY_SCHEMAS),
    variant_name="A_zero_shot",
    model_id=MODEL_ID,
)
save_jsonl(preds_a, PRED_DIR / "variant_A_zero_shot_val.jsonl")

## 6. Variant B — Zero-shot + constrained JSON decoding

In [ ]:
import outlines

outlines_model = outlines.from_transformers(model, tokenizer)

json_generators = {}
for cat, schema in JSON_SCHEMAS.items():
    json_generators[cat] = outlines.generate.json(outlines_model, schema)
    print(f"  Built JSON generator for: {cat}")

print(f"\n{len(json_generators)} generators ready.")

In [ ]:
import time
from tqdm.auto import tqdm

def run_constrained_inference(records: list[dict]) -> list[dict]:
    """Run Variant B: zero-shot with outlines constrained JSON decoding."""
    predictions = []

    for rec in tqdm(records, desc="Variant B (constrained)"):
        messages = build_messages_zero_shot(rec, CATEGORY_SCHEMAS)
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        generator = json_generators[rec["category"]]

        t0 = time.perf_counter()
        try:
            result = generator(text, max_tokens=512)
            latency = time.perf_counter() - t0
            if isinstance(result, dict):
                pred_json = result
            elif isinstance(result, str):
                pred_json = json.loads(result)
            else:
                pred_json = dict(result) if hasattr(result, "__dict__") else json.loads(str(result))
            parse_error = None
            raw_output = json.dumps(pred_json)
        except Exception as e:
            latency = time.perf_counter() - t0
            pred_json = None
            parse_error = f"Constrained generation failed: {e}"
            raw_output = str(e)

        predictions.append({
            "id": rec["id"],
            "category": rec["category"],
            "variant": "B_constrained",
            "model": MODEL_ID,
            "gold_json": rec["gold_json"],
            "pred_json": pred_json,
            "raw_output": raw_output,
            "parse_error": parse_error,
            "input_tokens": -1,
            "output_tokens": -1,
            "latency_s": round(latency, 3),
        })

    n_parsed = sum(1 for p in predictions if p["pred_json"] is not None)
    avg_latency = sum(p["latency_s"] for p in predictions) / len(predictions)
    print(f"\nB_constrained: {n_parsed}/{len(predictions)} parsed "
          f"({n_parsed/len(predictions)*100:.1f}%), avg latency {avg_latency:.2f}s")
    return predictions

preds_b = run_constrained_inference(val)
save_jsonl(preds_b, PRED_DIR / "variant_B_constrained_val.jsonl")

## 7. Variant C — Few-shot (3 examples) on full val set

In [ ]:
from ave.prompts.templates import build_messages_few_shot

preds_c = run_inference(
    model, tokenizer,
    records=val,
    message_builder=lambda rec: build_messages_few_shot(rec, CATEGORY_SCHEMAS, train_by_cat, n_examples=3),
    variant_name="C_few_shot_3",
    model_id=MODEL_ID,
)
save_jsonl(preds_c, PRED_DIR / "variant_C_few_shot_3_val.jsonl")

## 8. Quick comparison across variants

In [ ]:
def quick_metrics(predictions: list[dict]) -> dict:
    n = len(predictions)
    n_parsed = sum(1 for p in predictions if p["pred_json"] is not None)

    n_schema_ok = 0
    for p in predictions:
        if p["pred_json"] is not None:
            expected_keys = set(CATEGORY_SCHEMAS[p["category"]])
            actual_keys = set(p["pred_json"].keys())
            if expected_keys == actual_keys:
                n_schema_ok += 1

    n_exact = sum(1 for p in predictions
                  if p["pred_json"] is not None and p["pred_json"] == p["gold_json"])

    latencies = [p["latency_s"] for p in predictions]

    return {
        "n": n,
        "valid_json_pct": round(n_parsed / n * 100, 1),
        "schema_match_pct": round(n_schema_ok / n * 100, 1),
        "exact_match_pct": round(n_exact / n * 100, 1),
        "avg_latency_s": round(sum(latencies) / n, 2),
        "total_time_s": round(sum(latencies), 1),
    }


all_preds = {
    "A - Zero-shot": preds_a,
    "B - Constrained": preds_b,
    "C - Few-shot (3)": preds_c,
}

comparison_rows = []
for name, preds in all_preds.items():
    metrics = quick_metrics(preds)
    metrics["variant"] = name
    comparison_rows.append(metrics)

comp_df = pd.DataFrame(comparison_rows).set_index("variant")
print("Quick comparison across variants (val set):\n")
print(comp_df.to_string())

In [ ]:
# Per-category breakdown
for name, preds in all_preds.items():
    print(f"\n=== {name} — per-category valid JSON rate ===")
    cat_stats = defaultdict(lambda: {"total": 0, "parsed": 0, "schema_ok": 0})
    for p in preds:
        cat_stats[p["category"]]["total"] += 1
        if p["pred_json"] is not None:
            cat_stats[p["category"]]["parsed"] += 1
            expected = set(CATEGORY_SCHEMAS[p["category"]])
            if set(p["pred_json"].keys()) == expected:
                cat_stats[p["category"]]["schema_ok"] += 1

    for cat in CATEGORY_SCHEMAS:
        s = cat_stats[cat]
        parse_pct = s["parsed"] / s["total"] * 100 if s["total"] else 0
        schema_pct = s["schema_ok"] / s["total"] * 100 if s["total"] else 0
        print(f"  {cat:35s}  valid_json={parse_pct:5.1f}%  schema_match={schema_pct:5.1f}%  (n={s['total']})")

## 9. Inspect failure cases

In [ ]:
# Parse failures from Variant A
parse_failures = [p for p in preds_a if p["pred_json"] is None]
print(f"Variant A parse failures: {len(parse_failures)} / {len(preds_a)}\n")

for p in parse_failures[:5]:
    print(f"  ID={p['id']} ({p['category']})")
    print(f"  Error: {p['parse_error']}")
    print(f"  Raw output (first 300 chars):")
    print(f"    {p['raw_output'][:300]}")
    print()

In [ ]:
# Schema mismatches from Variant A (parsed but wrong keys)
schema_mismatches = []
for p in preds_a:
    if p["pred_json"] is not None:
        expected = set(CATEGORY_SCHEMAS[p["category"]])
        actual = set(p["pred_json"].keys())
        if expected != actual:
            schema_mismatches.append({
                **p,
                "missing_keys": expected - actual,
                "extra_keys": actual - expected,
            })

print(f"Variant A schema mismatches: {len(schema_mismatches)} / {len(preds_a)}\n")

for p in schema_mismatches[:5]:
    print(f"  ID={p['id']} ({p['category']})")
    if p["missing_keys"]:
        print(f"    Missing: {p['missing_keys']}")
    if p["extra_keys"]:
        print(f"    Extra:   {p['extra_keys']}")
    print()

## 10. Latency and token usage summary

In [ ]:
latency_rows = []
for name, preds in all_preds.items():
    for p in preds:
        latency_rows.append({
            "variant": name,
            "category": p["category"],
            "latency_s": p["latency_s"],
            "input_tokens": p["input_tokens"],
            "output_tokens": p["output_tokens"],
        })

lat_df = pd.DataFrame(latency_rows)

print("Latency statistics per variant:\n")
print(lat_df.groupby("variant")["latency_s"].describe().round(2))
print()

# Token usage (A and C — B doesn't track tokens)
for name in ["A - Zero-shot", "C - Few-shot (3)"]:
    subset = lat_df[lat_df["variant"] == name]
    print(f"{name}:")
    print(f"  Input tokens:  mean={subset['input_tokens'].mean():.0f}, "
          f"max={subset['input_tokens'].max()}")
    print(f"  Output tokens: mean={subset['output_tokens'].mean():.0f}, "
          f"max={subset['output_tokens'].max()}")
    print()

## 11. List prediction files

In [ ]:
print("Prediction files:")
for p in sorted(PRED_DIR.iterdir()):
    if p.suffix == ".jsonl":
        n_lines = sum(1 for _ in open(p))
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:50s} {n_lines:>4} records  {size_kb:>8.1f} KB")

## 12. (Optional) Copy predictions to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DEST = Path("/content/drive/MyDrive/ave-predictions")
DRIVE_DEST.mkdir(parents=True, exist_ok=True)

for p in sorted(PRED_DIR.iterdir()):
    if p.suffix == ".jsonl":
        shutil.copy(p, DRIVE_DEST / p.name)
        print(f"  Copied {p.name} -> {DRIVE_DEST / p.name}")

print(f"\nAll predictions saved to Google Drive: {DRIVE_DEST}")

## Summary

**Model:** `Qwen/Qwen3-4B-Instruct-2507`

**Variants run on val set:**
- **A — Zero-shot:** greedy decoding, no constraints
- **B — Constrained:** same prompt + `outlines` JSON Schema decoding
- **C — Few-shot (3):** 3 same-category examples from train split, greedy decoding

**Output files (in `predictions/`):**
- `variant_A_zero_shot_val.jsonl`
- `variant_B_constrained_val.jsonl`
- `variant_C_few_shot_3_val.jsonl`

**Next steps:**
- Notebook 06: fine-tuning (Variant D)
- Notebook 07: proprietary API baseline (Variant E)
- Notebook 08: full evaluation with field-level F1, null handling, etc.
- Notebook 09: results comparison and visualization